# 5-2. 트랜스포머 블록 조립하기

> **이 노트북에서 답할 질문**
> 어텐션 하나만으로는 왜 부족하고, 나머지 부품들은 각각 무슨 일을 하는가?

5-1에서 어텐션 헤드를 만들었습니다. 하지만 GPT는 그것 하나로 되어 있지 않습니다.
블록 하나는 이렇게 생겼습니다.

```
        x ──────────────────┐
        │                   │
   LayerNorm                │
        │                   │
  Multi-Head Attention      │  ← 토큰들끼리 정보를 "섞는" 단계
        │                   │
        + ←─────────────────┘   (잔차 연결)
        │
        ├───────────────────┐
        │                   │
   LayerNorm                │
        │                   │
   FeedForward              │  ← 각 토큰이 홀로 "생각하는" 단계
        │                   │
        + ←─────────────────┘
        │
        출력
```

부품이 네 개입니다. 하나씩, **왜 필요한지**를 확인하며 붙여 나갑니다.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from common import set_seed, get_device, count_params, human

set_seed(1234)
device = get_device()

## 1. 멀티헤드 — 여러 관계를 동시에

어텐션 헤드 하나는 **한 종류의 관계**만 잡습니다.
하지만 문장 안의 관계는 여러 종류입니다.

> `그 고양이가 생선을 먹었다`
> - `먹었다` → `고양이가` : 누가 했는가 (주어 관계)
> - `먹었다` → `생선을` : 무엇을 했는가 (목적어 관계)
> - `고양이가` → `그` : 무엇을 꾸미는가

한 헤드로 이걸 다 하려면 무리입니다. **헤드를 여러 개 두고 나중에 합칩니다.**

### 중요한 설계: 차원을 나눠 갖습니다

헤드를 8개 두면서 각 헤드가 `n_embd` 전체를 쓰면 계산량이 8배가 됩니다.
대신 **`head_size = n_embd // n_head`** 로 쪼갭니다. **전체 계산량은 그대로**입니다.

In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        att = (q @ k.transpose(-2, -1)) * k.shape[-1] ** -0.5
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        att = self.drop(F.softmax(att, dim=-1))
        return att @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd 는 n_head 로 나누어떨어져야 합니다"
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout)
                                    for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)    # 합친 뒤 섞어 주는 층
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B,T,n_embd)
        return self.drop(self.proj(out))

n_embd, block_size = 64, 16
for n_head in (1, 2, 4, 8):
    m = MultiHeadAttention(n_embd, n_head, block_size)
    print(f"헤드 {n_head}개 (head_size={n_embd//n_head:>2}) -> 파라미터 {human(count_params(m))}")
print(f"{chr(10)}헤드 수를 늘려도 파라미터가 거의 같습니다 -- 차원을 나눠 갖기 때문입니다.")

## 2. 피드포워드 — 혼자 생각하는 단계

어텐션은 **토큰들 사이에 정보를 옮기는** 일만 합니다. 가중 평균이니까요.
옮겨 온 정보를 **가공**하는 단계가 따로 필요합니다.

그게 피드포워드 네트워크(FFN)입니다. 각 토큰에 **독립적으로** 적용됩니다.

$$\text{FFN}(x) = W_2 \,\text{ReLU}(W_1 x + b_1) + b_2$$

### 왜 4배로 부풀리나

관례적으로 중간 차원을 `4 * n_embd` 로 잡습니다.
좁은 공간에서는 표현하기 어려운 패턴을 **넓은 공간으로 올려서 처리하고 다시 내려옵니다.**

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, mult=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, mult * n_embd),   # 넓히고
            nn.ReLU(),                          # 비선형
            nn.Linear(mult * n_embd, n_embd),   # 되돌리고
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

ff = FeedForward(64)
mha = MultiHeadAttention(64, 4, 16)
print(f"MultiHeadAttention 파라미터: {human(count_params(mha))}")
print(f"FeedForward        파라미터: {human(count_params(ff))}")
print(f"{chr(10)}FFN이 어텐션보다 파라미터가 많습니다. 실제 LLM에서도 그렇습니다.")
print("'모델 지식의 대부분은 FFN에 저장된다'는 연구들이 여기서 나옵니다.")

## 3. 잔차 연결 — 깊게 쌓기 위한 필수품

4-1에서 예고한 그것입니다. **$y = f(x) + x$**. 파라미터 0개.

왜 필요할까요? 층을 깊게 쌓으면 역전파 과정에서 기울기가 **곱셈으로 누적**되어
0으로 사라지거나(vanishing) 폭발합니다.

잔차 연결이 있으면 기울기가 **덧셈 경로**로 곧장 흘러갑니다. 직접 확인해 봅시다.

In [ ]:
def grad_through(depth, residual):
    """depth 개의 층을 통과시킨 뒤, 입력에 도달하는 기울기 크기를 잽니다."""
    set_seed(0)
    x = torch.randn(1, 64, requires_grad=True)
    layers = [nn.Sequential(nn.Linear(64, 64), nn.ReLU()) for _ in range(depth)]
    h = x
    for layer in layers:
        h = h + layer(h) if residual else layer(h)
    h.sum().backward()
    return x.grad.abs().mean().item()

print(f"{'깊이':>6} | {'잔차 없음':>14} | {'잔차 있음':>14}")
print("-" * 40)
for depth in (2, 8, 20, 50):
    print(f"{depth:>6} | {grad_through(depth, False):>14.3e} | {grad_through(depth, True):>14.3e}")
print(f"{chr(10)}잔차가 없으면 깊어질수록 기울기가 소멸합니다. 앞쪽 층이 학습되지 않습니다.")

> **핵심 직관**: 잔차 연결은 각 층이 "출력을 처음부터 만드는" 대신
> **"입력을 조금 고치는"** 역할만 하게 만듭니다. 훨씬 쉬운 일이죠.
> 이것이 100층 넘는 신경망이 가능해진 결정적 이유입니다.

## 4. 레이어 정규화 — 값의 크기를 고르게

층을 지날수록 값의 분포가 제멋대로 커지거나 작아집니다.
LayerNorm은 **각 토큰의 벡터를 평균 0, 분산 1로 맞춥니다.**

$$\hat{x} = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

$\gamma, \beta$ 는 학습되는 값이라, 필요하면 모델이 정규화를 "되돌릴" 수도 있습니다.

In [ ]:
ln = nn.LayerNorm(64)
x = torch.randn(2, 5, 64) * 10 + 3      # 평균 3, 표준편차 10 쯤 되는 값

print(f"정규화 전: 평균 {x.mean():.3f},  표준편차 {x.std():.3f}")
y = ln(x)
print(f"정규화 후: 평균 {y.mean():.3f},  표준편차 {y.std():.3f}")

print(f"{chr(10)}주의 - LayerNorm은 '토큰별로' 정규화합니다 (배치 전체가 아님):")
print(f"  토큰 하나의 평균: {y[0, 0].mean().item():.2e},  표준편차: {y[0, 0].std().item():.4f}")
print(f"{chr(10)}BatchNorm과의 차이: BatchNorm은 배치 방향으로 정규화해서")
print("배치 크기와 시퀀스 길이에 의존합니다. 그래서 언어 모델에는 LayerNorm을 씁니다.")

### Pre-LN vs Post-LN — 현대적 선택

원논문(2017)은 **Post-LN** 이었습니다: `x + Attention(x)` 를 한 뒤 정규화.

```
Post-LN:  x = LayerNorm(x + Attention(x))     # 원논문
Pre-LN :  x = x + Attention(LayerNorm(x))     # 현대 표준
```

차이가 결정적입니다. **Pre-LN은 잔차 경로에 정규화가 끼어들지 않습니다.**
입력에서 출력까지 아무 방해 없는 덧셈 고속도로가 뚫립니다.

Post-LN은 학습률 워밍업을 세심하게 하지 않으면 발산하기 쉽습니다.
**GPT-2 이후로는 사실상 전부 Pre-LN입니다.**

## 5. 블록 완성

In [ ]:
class Block(nn.Module):
    """트랜스포머 블록 하나 (Pre-LN 방식)."""
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))    # 섞기
        x = x + self.ffwd(self.ln2(x))    # 생각하기
        return x

blk = Block(n_embd=64, n_head=4, block_size=16)
x = torch.randn(2, 10, 64)
print(f"입력 {tuple(x.shape)} -> 출력 {tuple(blk(x).shape)}   (모양이 같습니다!)")
print(f"블록 하나의 파라미터: {human(count_params(blk))}")
print(f"{chr(10)}입출력 모양이 같다는 건 -> 블록을 그냥 쌓을 수 있다는 뜻입니다.")

In [ ]:
# 블록을 쌓아 보기
per_block = count_params(Block(768, 12, 1024))
print(f"블록 1개 (n_embd=768, n_head=12) = {human(per_block)}{chr(10)}")
for n_layer in (1, 4, 12, 48):
    print(f"n_layer={n_layer:>3} -> 블록 합계 {human(per_block * n_layer)}")

print(f"{chr(10)}참고: GPT-2 small = 12층 x 768차원 = 약 124M 파라미터")
print("     (여기에 임베딩과 출력층이 더해집니다)")

## 6. 위치 정보 — 빠진 조각 하나

여기까지 오면 블록이 완성된 것 같지만, **치명적인 구멍**이 있습니다.

어텐션은 **순서를 모릅니다.** 가중 평균은 순서와 무관하니까요.
`나는 너를 좋아해` 와 `너를 나는 좋아해` 를 구분하지 못합니다.

직접 확인해 봅시다.

In [ ]:
set_seed(0)
blk = Block(32, 4, 8)
blk.eval()

x = torch.randn(1, 4, 32)
perm = torch.tensor([1, 0, 2, 3])       # 앞 두 토큰의 순서를 바꿈

with torch.no_grad():
    out_orig = blk(x)
    out_perm = blk(x[:, perm])

# 마지막 토큰의 출력은 "앞의 모든 토큰의 집합"에만 의존해야 함 (순서 무관)
print(f"마지막 토큰 출력이 같은가? {torch.allclose(out_orig[:, -1], out_perm[:, -1], atol=1e-5)}")
print(f"차이: {(out_orig[:, -1] - out_perm[:, -1]).abs().max().item():.2e}")
print(f"{chr(10)}-> 토큰 순서를 바꿔도 결과가 같습니다. 어텐션은 순서를 모릅니다.")

### 해결: 위치 임베딩

**단어 임베딩에 위치 임베딩을 더해 줍니다.**

$$x = E_{\text{token}}[w_t] + E_{\text{pos}}[t]$$

이러면 같은 단어라도 위치가 다르면 다른 벡터가 되어, 어텐션이 순서를 구별할 수 있습니다.

In [ ]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)    # 학습되는 위치 임베딩

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        return self.tok_emb(idx) + self.pos_emb(pos)       # 브로드캐스트로 더해짐

emb = Embeddings(vocab_size=100, n_embd=32, block_size=8)
idx = torch.tensor([[5, 7, 5]])          # 같은 토큰 5가 0번과 2번 자리에
e = emb(idx)
print(f"같은 토큰(5)이지만 위치가 달라서 벡터가 다른가?"
      f" {not torch.allclose(e[0, 0], e[0, 2])}")
print(f"두 벡터의 차이 크기: {(e[0,0] - e[0,2]).abs().mean().item():.4f}")
print(f"{chr(10)}이 차이가 바로 위치 임베딩이 만든 것입니다.")

### 위치 정보의 세 가지 방식

| 방식 | 설명 | 쓰는 곳 |
|---|---|---|
| **학습된 절대 위치** | `nn.Embedding(block_size, n_embd)` | GPT-2, BERT |
| **사인/코사인 고정** | 삼각함수로 계산, 학습 불필요 | 원논문(2017) |
| **RoPE (회전)** | Q·K를 위치만큼 회전 | Llama, Qwen 등 현대 모델 |

학습된 절대 위치의 치명적 한계: **`block_size` 를 넘는 위치는 아예 임베딩이 없습니다.**
학습 때 1024로 했으면 2048 토큰을 넣을 수 없습니다.

RoPE는 이 문제를 크게 완화합니다 — 5-3에서 다룹니다.

---

## 정리

**트랜스포머 블록의 부품과 역할**

| 부품 | 역할 | 없으면 |
|---|---|---|
| 멀티헤드 어텐션 | 토큰 간 정보 이동 | 문맥을 못 봄 |
| 피드포워드 | 토큰별 정보 가공 | 표현력 부족 |
| 잔차 연결 | 기울기 고속도로 | 깊게 못 쌓음 |
| LayerNorm | 값 크기 안정화 | 학습 불안정 |
| 위치 임베딩 | 순서 정보 | 어순을 구별 못 함 |

**핵심 구조**

```python
x = x + attn(ln1(x))   # 섞기
x = x + ffwd(ln2(x))   # 생각하기
```

이 두 줄이 GPT의 심장입니다. 나머지는 이걸 N번 반복하는 것뿐입니다.

## 연습

1. `Block` 을 Post-LN 방식(`x = ln1(x + attn(x))`)으로 바꾸고 5-2의 기울기 실험을 다시 해 보세요.
2. FFN의 `mult` 를 1, 4, 8로 바꿔 파라미터 비중이 어떻게 변하는지 보세요.
3. 위치 임베딩을 빼고 6절 실험을 다시 돌려, 순서 구별이 사라지는지 확인하세요.

## 다음

→ `0503_ModernRecipe.ipynb` : 2017년 원논문과 2025년 모델의 차이